In [4]:
!pip install -q simpy mesa scikit-learn pandas numpy matplotlib networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.1/275.1 kB 5.5 MB/s eta 0:00:00


In [3]:
!pip install simpy
import random
import statistics
import simpy


RANDOM_SEED = 42
SIMULATION_TIME = 100
PACKET_INTERVAL = 1.5

random.seed(RANDOM_SEED)


class Packet:
    """Represents one network packet."""

    def __init__(self, packet_id, creation_time):
        self.packet_id = packet_id
        self.creation_time = creation_time


class Router:
    """Processes packets and forwards them to the next router."""

    def __init__(self, env, name, processing_delay, next_router=None):
        self.env = env
        self.name = name
        self.processing_delay = processing_delay
        self.next_router = next_router

        # One packet can be processed at a time.
        self.processor = simpy.Resource(env, capacity=1)

        self.processed_packets = 0
        self.total_queue_wait = 0

    def receive_packet(self, packet):
        arrival_time = self.env.now

        with self.processor.request() as request:
            yield request

            queue_wait = self.env.now - arrival_time
            self.total_queue_wait += queue_wait

            print(
                f"Time {self.env.now:6.2f}: "
                f"Packet {packet.packet_id:3d} begins processing at "
                f"{self.name}; queue wait = {queue_wait:.2f}"
            )

            # Add a small amount of random variation to processing time.
            delay = random.uniform(
                self.processing_delay * 0.8,
                self.processing_delay * 1.2
            )

            yield self.env.timeout(delay)

            self.processed_packets += 1

            print(
                f"Time {self.env.now:6.2f}: "
                f"Packet {packet.packet_id:3d} leaves {self.name}"
            )

        if self.next_router is not None:
            yield self.env.process(
                self.next_router.receive_packet(packet)
            )
        else:
            # The last router delivers the packet.
            packet_latency = self.env.now - packet.creation_time
            network_statistics["latencies"].append(packet_latency)
            network_statistics["delivered"] += 1

            print(
                f"Time {self.env.now:6.2f}: "
                f"Packet {packet.packet_id:3d} delivered; "
                f"total latency = {packet_latency:.2f}"
            )


def packet_generator(env, first_router):
    """Creates packets at random intervals."""

    packet_id = 1

    while True:
        packet = Packet(packet_id, env.now)
        network_statistics["generated"] += 1

        print(
            f"\nTime {env.now:6.2f}: "
            f"Packet {packet.packet_id:3d} generated"
        )

        env.process(first_router.receive_packet(packet))

        packet_id += 1

        # Exponential arrival time simulates irregular network traffic.
        next_arrival = random.expovariate(1 / PACKET_INTERVAL)
        yield env.timeout(next_arrival)


network_statistics = {
    "generated": 0,
    "delivered": 0,
    "latencies": []
}

env = simpy.Environment()

# Build the router chain from the destination backward.
router3 = Router(
    env=env,
    name="Router 3",
    processing_delay=0.8,
    next_router=None
)

router2 = Router(
    env=env,
    name="Router 2",
    processing_delay=1.0,
    next_router=router3
)

router1 = Router(
    env=env,
    name="Router 1",
    processing_delay=0.6,
    next_router=router2
)

env.process(packet_generator(env, router1))
env.run(until=SIMULATION_TIME)


print("\n" + "=" * 55)
print("SIMULATION RESULTS")
print("=" * 55)

generated = network_statistics["generated"]
delivered = network_statistics["delivered"]
latencies = network_statistics["latencies"]

print(f"Simulation time:          {SIMULATION_TIME} time units")
print(f"Packets generated:        {generated}")
print(f"Packets delivered:        {delivered}")
print(f"Packets still in network: {generated - delivered}")

if latencies:
    average_latency = statistics.mean(latencies)
    minimum_latency = min(latencies)
    maximum_latency = max(latencies)
    throughput = delivered / SIMULATION_TIME

    print(f"Average latency:          {average_latency:.2f}")
    print(f"Minimum latency:          {minimum_latency:.2f}")
    print(f"Maximum latency:          {maximum_latency:.2f}")
    print(f"Throughput:               {throughput:.3f} packets/time unit")

print("\nROUTER STATISTICS")

for router in [router1, router2, router3]:
    average_wait = (
        router.total_queue_wait / router.processed_packets
        if router.processed_packets > 0
        else 0
    )

    print(
        f"{router.name}: "
        f"processed={router.processed_packets}, "
        f"average queue wait={average_wait:.2f}"
    )



Time   0.00: Packet   1 generated
Time   0.00: Packet   1 begins processing at Router 1; queue wait = 0.00
Time   0.49: Packet   1 leaves Router 1
Time   0.49: Packet   1 begins processing at Router 2; queue wait = 0.00
Time   1.40: Packet   1 leaves Router 2
Time   1.40: Packet   1 begins processing at Router 3; queue wait = 0.00

Time   1.53: Packet   2 generated
Time   1.53: Packet   2 begins processing at Router 1; queue wait = 0.00
Time   2.11: Packet   1 leaves Router 3
Time   2.11: Packet   1 delivered; total latency = 2.11
Time   2.17: Packet   2 leaves Router 1
Time   2.17: Packet   2 begins processing at Router 2; queue wait = 0.00
Time   3.33: Packet   2 leaves Router 2
Time   3.33: Packet   2 begins processing at Router 3; queue wait = 0.00

Time   3.53: Packet   3 generated
Time   3.53: Packet   3 begins processing at Router 1; queue wait = 0.00
Time   4.00: Packet   2 leaves Router 3
Time   4.00: Packet   2 delivered; total latency = 2.47
Time   4.02: Packet   3 leaves R